In [12]:
import sqlite3
import pandas as pd

# Carregando a base para treino 

conn = sqlite3.connect("..\data\credit_risk.db")
df= pd.read_sql("SELECT * from credito_integrado",conn)
print(df.dtypes)
print(df.head)

person_age                      int64
person_income                 float64
person_home_ownership             str
person_emp_length              object
loan_intent                       str
loan_grade                        str
loan_amnt                     float64
loan_int_rate                  object
loan_status                     int64
loan_percent_income           float64
cb_person_default_on_file         str
cb_person_cred_hist_length      int64
date                            int64
year                            int64
month                           int64
unnamed_0                         str
unnamed_1                         str
ipca                          float64
no                            float64
dez_93_100                    float64
no_1                          float64
n12                           float64
ipca_tipo                         str
mes_num                         int64
data_ref                          str
selic_media                   float64
dtype: objec

In [13]:
# Random Forest 

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score 
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

In [14]:
features = [
    'person_age', 'person_income', 'person_emp_length', 'loan_amnt',
    'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length',
    'cb_person_default_on_file'
]

target = 'loan_status'

In [15]:
# Tratamento

df_model_rf = df[features + [target]].copy()

# converter Y/N -> 1/0 
df_model_rf['cb_person_default_on_file'] = df_model_rf['cb_person_default_on_file'].map({'Y': 1, 'N': 0})

# garantir numérico 
for col in df_model_rf.columns:
    df_model_rf[col] = pd.to_numeric(df_model_rf[col], errors = 'coerce')

df_model_rf.dropna(inplace=True)

# Binarios
df_model_rf = df_model_rf[df_model_rf[target].isin([0, 1])]
df_model_rf[target] = df_model_rf[target].astype(int)

print(df_model_rf.shape)
df_model_rf.head()

(28638, 9)


,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,cb_person_default_on_file,loan_status
0,22,59000.0,123.0,35000.0,16.02,0.59,3,1,1
1,21,9600.0,5.0,1000.0,11.14,0.10,2,0,0
2,25,9600.0,1.0,5500.0,12.87,0.57,3,0,1
3,23,65500.0,4.0,35000.0,15.23,0.53,2,0,1
4,24,54400.0,8.0,35000.0,14.27,0.55,4,1,1


In [16]:
# Split

X_rf = df_model_rf[features]
y_rf = df_model_rf[target]

X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(
    X_rf, y_rf, 
    test_size= 0.2,
    random_state=42, 
    stratify=y_rf
)

print("Treino:", X_train_rf.shape, y_train_rf.shape)
print("Teste:", X_test_rf.shape, y_test_rf.shape)

Treino: (22910, 8) (22910,)
Teste: (5728, 8) (5728,)


In [17]:
# Modelo 

modelo_rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_split=20,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

In [18]:
# Validacao (cross-val)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores= cross_val_score(
    modelo_rf,
    X_train_rf,
    y_train_rf, 
    cv=cv,
    scoring='roc_auc'
)

print(f"AUC-ROC CV: {scores.mean():.4f} +- {scores.std():.4f}")

AUC-ROC CV: 0.8831 +- 0.0062


In [19]:
# Treino Final

modelo_rf.fit(X_train_rf, y_train_rf)
print("deucertin:)")

deucertin:)


In [20]:
# Teste

y_pred_rf = modelo_rf.predict(X_test_rf)
y_proba_rf = modelo_rf.predict_proba(X_test_rf)[:, 1]

auc_rf = roc_auc_score(y_test_rf, y_proba_rf)

print(f"\n AUC-ROC Test: {auc_rf:.4f}")
print("\nClassification Report:")
print(classification_report(y_test_rf, y_pred_rf))


 AUC-ROC Test: 0.8814

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.87      0.90      4487
           1       0.61      0.74      0.67      1241

    accuracy                           0.84      5728
   macro avg       0.77      0.81      0.78      5728
weighted avg       0.86      0.84      0.85      5728



In [21]:
# Matriz de confusão + relevancia

cm_rf = confusion_matrix(y_test_rf, y_pred_rf)
print("Matriz de confusão - Random Forest")
print(cm_rf)

importancias = pd.Series(
    modelo_rf.feature_importances_, 
    index=features
).sort_values(ascending=False)

print("\nImportância das variáveis:")
print(importancias)

Matriz de confusão - Random Forest
[[3898  589]
 [ 321  920]]

Importância das variáveis:
loan_int_rate                 0.310686
loan_percent_income           0.302442
person_income                 0.207490
loan_amnt                     0.071523
cb_person_default_on_file     0.042944
person_emp_length             0.030456
person_age                    0.019533
cb_person_cred_hist_length    0.014927
dtype: float64


In [22]:
# Resumo 

print("Resumo do modelo")
print(f"AUC-ROC CV: {scores.mean():.4f} +- {scores.std():.4f}")
print(f"AUC-ROC Test: {auc_rf:.4f}")

Resumo do modelo
AUC-ROC CV: 0.8831 +- 0.0062
AUC-ROC Test: 0.8814
